# Notebook 03 — AND-Gate Consolidation Simulation

Demonstrates the three-axis AND-gate model for addiction consolidation
and quantifies the non-linear protective advantage of multi-axis modulation.

**Axes**:  
1. DA salience (phasic burst / tonic ratio)  
2. NMDA plasticity (corticostriatal LTP probability)  
3. Reward contrast (positive RPE magnitude)  

**Key result**: At 50% modulation, triple-axis intervention reduces  
consolidation probability by f³ vs f for single-axis (4× more protective at f=0.5).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from neuropharm_sim.and_gate import (
    ANDGate, SalienceAxis, NMDAPlasticityAxis, RewardContrastAxis
)
from neuropharm_sim.visualization import (
    plot_and_gate_protection, plot_phase_diagram
)

## 1. Individual axis responses

In [ ]:
burst_range = np.linspace(0, 6, 200)
strength_range = np.linspace(0, 1, 200)

sal_ax   = SalienceAxis()
nmda_ax  = NMDAPlasticityAxis()
cont_ax  = RewardContrastAxis()

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))

ax1.plot(burst_range, sal_ax.activation(burst_range), lw=2, color="#762a83")
ax1.axvline(sal_ax.threshold, ls="--", color="gray", lw=1)
ax1.set_xlabel("Burst / tonic ratio")
ax1.set_ylabel("Axis activation")
ax1.set_title("DA Salience Axis")
ax1.set_ylim(0, 1.05)

ax2.plot(strength_range, nmda_ax.activation(strength_range), lw=2, color="#1a9641")
ax2.axvline(nmda_ax.threshold, ls="--", color="gray", lw=1)
ax2.set_xlabel("Synaptic input strength")
ax2.set_title("NMDA Plasticity Axis")
ax2.set_ylim(0, 1.05)

rpe_range = np.linspace(-0.5, 1.0, 200)
ax3.plot(rpe_range, cont_ax.activation(rpe_range + 0.5, np.full_like(rpe_range, 0.5)),
         lw=2, color="#d6604d")
ax3.axvline(0, ls="--", color="gray", lw=1, label="RPE = 0")
ax3.set_xlabel("RPE (actual − predicted)")
ax3.set_title("Reward Contrast Axis")
ax3.set_ylim(0, 1.05)
ax3.legend(fontsize=9)

fig.suptitle("Individual Axis Transfer Functions", fontweight="bold")
fig.tight_layout()
plt.show()

## 2. Single-session AND-gate evaluation

In [ ]:
gate = ANDGate(consolidation_threshold=0.1)

# High-risk scenario: strong phasic burst, potentiated synapses, large RPE
result = gate.evaluate(
    burst_to_tonic=4.0,
    synaptic_strength=0.85,
    actual_reward=0.95,
    predicted_reward=0.1,
)

print("High-risk scenario")
print(f"  DA salience activation : {float(result.salience_activation):.3f}")
print(f"  NMDA plasticity act.   : {float(result.nmda_activation):.3f}")
print(f"  Reward contrast act.   : {float(result.contrast_activation):.3f}")
print(f"  AND-gate probability   : {float(result.gate_probability):.3f}")
print(f"  Above threshold (0.1)  : {bool(result.above_threshold)}")

## 3. Protection analysis — super-additivity

In [ ]:
modulation = np.linspace(0, 1, 60)
pa = gate.protection_analysis(
    burst_to_tonic=4.0,
    synaptic_strength=0.85,
    actual_reward=0.95,
    predicted_reward=0.1,
    modulation_levels=modulation,
)

fig, ax = plot_and_gate_protection(pa)
plt.show()

## 4. Quantitative protection table

In [ ]:
test_levels = [0.8, 0.6, 0.4, 0.2, 0.0]
rows = []
for level in test_levels:
    idx = np.argmin(np.abs(modulation - level))
    baseline_p = pa["baseline"][0]
    single_p = pa["salience_only"][idx]
    triple_p = pa["triple"][idx]
    rows.append({
        "Modulation factor": f"{level:.1f}",
        "Baseline P": f"{baseline_p:.4f}",
        "Single-axis P": f"{single_p:.4f}",
        "Triple-axis P": f"{triple_p:.4f}",
        "Ratio (single/triple)": f"{single_p/max(triple_p,1e-9):.1f}×",
        "% reduction (triple vs baseline)": f"{(1-triple_p/baseline_p)*100:.1f}%" if baseline_p > 0 else "N/A",
    })

pd.DataFrame(rows).set_index("Modulation factor")

## 5. Phase diagram: salience × NMDA space

In [ ]:
from neuropharm_sim.visualization import plot_phase_diagram

burst_vals    = np.linspace(0.2, 6.0, 60)
strength_vals = np.linspace(0.0, 1.0, 60)

grid = gate.phase_diagram(
    axis1_values=burst_vals,
    axis2_values=strength_vals,
    fixed_contrast_activation=0.85,
)

fig, ax = plot_phase_diagram(
    grid, burst_vals, strength_vals,
    consolidation_threshold=0.1,
    xlabel="DA salience (burst/tonic ratio)",
    ylabel="Corticostriatal strength (NMDA axis)",
    title="AND-Gate Phase Diagram (RPE contrast fixed at 0.85)",
)
plt.show()

## 6. Effect of NMDA modulation (memantine-analogue)

In [ ]:
nmda_levels = [1.0, 0.7, 0.4, 0.1]
colors = ["#d73027", "#fc8d59", "#91bfdb", "#2166ac"]

fig, ax = plt.subplots(figsize=(8, 5))
for level, color in zip(nmda_levels, colors):
    gate_mod = ANDGate(
        nmda_axis=NMDAPlasticityAxis(nmda_modulation=level)
    )
    burst_range = np.linspace(0, 6, 100)
    probs = np.array([
        gate_mod._gate_scalar(
            b, 0.85, 0.95, 0.1,
            sal_mod=1.0, nmda_mod=level,
            con_prec=gate.contrast_axis.expectation_precision
        )
        for b in burst_range
    ])
    ax.plot(burst_range, probs, lw=2, color=color,
            label=f"NMDA availability = {level:.0%}")

ax.set_xlabel("Phasic DA burst / tonic ratio")
ax.set_ylabel("Consolidation probability")
ax.set_title("NMDA Modulation Shifts AND-Gate Threshold", fontweight="bold")
ax.legend(fontsize=10)
ax.axhline(0.1, ls="--", color="gray", alpha=0.5, label="threshold")
plt.tight_layout()
plt.show()